# Recovering a shop order stream

Name: Debottam Sarkar
Registration number: 202400303

This experiment uses Spark Structured Streaming to read incoming order files and write a transaction-safe SQLite ledger. It deliberately fails after the sink has committed a batch, then resumes from the same checkpoint. The checks compare order IDs and monetary totals with the original generated input.

Run all cells in order in a fresh Colab Python runtime. The orders are simulated; no account, API key or external dataset is required.

In [1]:
%pip -q install pyspark==4.0.1 pandas==2.2.3


[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: /tmp/dwbda-notebooks-oSWVK5/test-env/bin/python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## Set up an isolated run

Every full run creates a new working directory, so repeating the notebook does not delete an earlier checkpoint. Within this run, the input folder, database and checkpoint paths stay fixed across failures and restarts.

Colab's local disk is temporary. This demonstration recovers a failed query while the runtime files survive. Recovery after deletion of the runtime would need storage that survives independently of the Colab VM.

In [2]:
import json
import os
import sqlite3
import sys
from contextlib import closing
from datetime import datetime, timedelta
from pathlib import Path
from tempfile import mkdtemp

import pandas as pd
from IPython.display import display
from pyspark.errors import StreamingQueryException
from pyspark.sql import SparkSession, functions as F

os.environ["PYSPARK_PYTHON"] = sys.executable
root = Path(mkdtemp(prefix="order_stream_"))
incoming = root / "incoming"
staging = root / "staging"
checkpoint = root / "checkpoint"
ledger = root / "orders.sqlite"
armed = root / "failure_armed"
failed_once = root / "failure_observed"
incoming.mkdir()
staging.mkdir()

spark = (
    SparkSession.builder.master("local[2]")
    .appName("OrderLedgerRecovery")
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark:", spark.version)

with closing(sqlite3.connect(ledger)) as connection:
    connection.executescript("""
        CREATE TABLE orders (
            order_id TEXT PRIMARY KEY,
            placed_at TEXT NOT NULL,
            department TEXT NOT NULL,
            items INTEGER NOT NULL,
            unit_price_cents INTEGER NOT NULL
        );
        CREATE TABLE processed_batches (
            batch_id INTEGER PRIMARY KEY,
            input_rows INTEGER NOT NULL
        );
    """)
    connection.commit()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/16 15:58:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark: 4.0.1


## Publish order files

Three deliveries contain 18 distinct orders. Each includes a duplicate and an invalid record. A staging file is moved into the watched directory only after it is completely written. Money is stored as integer cents, avoiding floating-point rounding in totals.

Delivery happens in separate cells so the query can be tested before and after a failure. `availableNow` runs Spark's streaming engine until the files currently available have been processed.

In [3]:
departments = ["Books", "Stationery", "Electronics"]
prices = [12500, 4500, 38900]
start_time = datetime(2026, 11, 1, 9, 0)

expected_orders = [
    {
        "order_id": f"ORD-{number + 1:03d}",
        "placed_at": (start_time + timedelta(seconds=10 * number)).isoformat(),
        "department": departments[number % 3],
        "items": 1 + (number // 3) % 3,
        "unit_price_cents": prices[number % 3]
    }
    for number in range(18)
]

def deliver(part, records, invalid_line):
    name = f"delivery_{part:02d}.json"
    target = incoming / name
    if target.exists():
        raise FileExistsError(f"{name} is already published. Start a fresh run to repeat delivery.")
    temporary = staging / name
    lines = [json.dumps(record) for record in records] + [invalid_line]
    temporary.write_text("\n".join(lines) + "\n", encoding="utf-8")
    temporary.replace(target)
    print(f"{name}: {len(lines)} incoming lines")

def read_table(statement):
    with closing(sqlite3.connect(ledger)) as connection:
        return pd.read_sql_query(statement, connection)

deliver(0, expected_orders[:6] + [expected_orders[1]], "{broken json")

delivery_00.json: 8 incoming lines


## Validate and de-duplicate

The schema exposes malformed JSON through `_bad_record`. Missing fields, unsupported departments and non-positive quantities or prices are also rejected. A two-minute event-time watermark bounds the streaming de-duplication state.

Order IDs identify logical events. The ledger also has a primary key on the ID, so a repeat cannot create a second order even after the watermark state has expired. Watermarks can discard sufficiently late data; this demonstration's repeats arrive within two minutes.

In [4]:
schema = """
    order_id STRING,
    placed_at TIMESTAMP,
    department STRING,
    items INT,
    unit_price_cents INT,
    _bad_record STRING
"""
source = (
    spark.readStream.schema(schema)
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_bad_record")
    .option("maxFilesPerTrigger", 1)
    .json(str(incoming))
)
valid_orders = source.where(
    F.col("_bad_record").isNull()
    & F.col("order_id").rlike("^ORD-[0-9]{3}$")
    & F.col("placed_at").isNotNull()
    & F.col("department").isin(departments)
    & (F.col("items") > 0)
    & (F.col("unit_price_cents") > 0)
)
unique_orders = (
    valid_orders.withWatermark("placed_at", "2 minutes")
    .dropDuplicatesWithinWatermark(["order_id"])
    .select("order_id", "placed_at", "department", "items", "unit_price_cents")
)

## Commit each batch atomically

`foreachBatch` can be called again for a batch after a failure. The transaction below writes the order rows and its batch ID together. An already committed batch is skipped on replay. If a failure happens before the SQLite transaction commits, both writes roll back.

The callback collects a small, bounded micro-batch into the driver. SQLite is suitable for this local exercise; a large distributed stream would use a sink designed for concurrent, partitioned writes.

In [5]:
def write_ledger(frame, batch_id):
    with closing(sqlite3.connect(ledger)) as connection:
        recorded = connection.execute(
            "SELECT 1 FROM processed_batches WHERE batch_id = ?", (int(batch_id),)
        ).fetchone()
        if recorded:
            frame.count()
            return

        printable = frame.select(
            'order_id',
            F.date_format('placed_at', "yyyy-MM-dd'T'HH:mm:ss").alias('placed_at'),
            'department', 'items', 'unit_price_cents'
        )
        rows = [
            (
                row.order_id, row.placed_at, row.department,
                int(row.items), int(row.unit_price_cents)
            )
            for row in printable.collect()
        ]
        with connection:
            connection.executemany(
                "INSERT OR IGNORE INTO orders VALUES (?, ?, ?, ?, ?)", rows
            )
            connection.execute(
                "INSERT INTO processed_batches VALUES (?, ?)", (int(batch_id), len(rows))
            )

    if rows and armed.exists() and not failed_once.exists():
        failed_once.touch()
        raise RuntimeError("Intentional interruption after durable sink commit")

def run_available():
    query = (
        unique_orders.writeStream.outputMode("append")
        .foreachBatch(write_ledger)
        .option("checkpointLocation", str(checkpoint))
        .trigger(availableNow=True)
        .start()
    )
    try:
        if not query.awaitTermination(120):
            raise TimeoutError("The query did not finish within two minutes.")
        return str(query.id)
    finally:
        if query.isActive:
            query.stop()

original_query_id = run_available()
first_delivery = read_table("SELECT * FROM orders ORDER BY order_id")
assert len(first_delivery) == 6
print("First delivery committed:", len(first_delivery), "unique orders")
display(first_delivery)

First delivery committed: 6 unique orders


,order_id,placed_at,department,items,unit_price_cents
0,ORD-001,2026-11-01T09:00:00,Books,1,12500
1,ORD-002,2026-11-01T09:00:10,Stationery,1,4500
2,ORD-003,2026-11-01T09:00:20,Electronics,1,38900
3,ORD-004,2026-11-01T09:00:30,Books,2,12500
4,ORD-005,2026-11-01T09:00:40,Stationery,2,4500
5,ORD-006,2026-11-01T09:00:50,Electronics,2,38900


## Interrupt after committing the second delivery

The sink commits the next six new orders and then raises an intentional exception before Spark can mark the micro-batch complete. This tests the gap between the external transaction and Spark's checkpoint commit. It is a controlled query failure, not a machine crash.

In [6]:
deliver(
    1, expected_orders[6:12] + [expected_orders[4]],
    json.dumps({"order_id": "INCOMPLETE"})
)
armed.touch()
spark.sparkContext.setLogLevel("OFF")
try:
    run_available()
except StreamingQueryException as error:
    if "Intentional interruption after durable sink commit" not in str(error):
        raise
    print("Expected interruption observed after the database transaction committed.")
else:
    raise AssertionError("The injected failure was not triggered.")
finally:
    spark.sparkContext.setLogLevel("ERROR")

assert failed_once.exists()
assert len(read_table("SELECT order_id FROM orders")) == 12
print("The ledger already holds 12 unique orders; Spark must retry the uncommitted batch.")

delivery_01.json: 8 incoming lines


Expected interruption observed after the database transaction committed.
The ledger already holds 12 unique orders; Spark must retry the uncommitted batch.


## Resume using the same checkpoint

The third delivery is published while the query is stopped. The restart replays unfinished work and then reads the new file. The database's batch log skips the committed transaction, and the restored state handles recent duplicate IDs.

In [7]:
deliver(2, expected_orders[12:] + [expected_orders[10]], "{incomplete json")
restarted_query_id = run_available()
assert restarted_query_id == original_query_id
print("Query identity restored from checkpoint:", restarted_query_id == original_query_id)

actual_orders = read_table("SELECT * FROM orders ORDER BY order_id")
expected_frame = pd.DataFrame(expected_orders).sort_values("order_id").reset_index(drop=True)
pd.testing.assert_frame_equal(actual_orders, expected_frame, check_dtype=False)
assert actual_orders.order_id.nunique() == 18

revenue = read_table("""
    SELECT department,
           COUNT(*) AS orders,
           SUM(items) AS units,
           SUM(items * unit_price_cents) AS revenue_cents
    FROM orders
    GROUP BY department
    ORDER BY department
""")
expected_revenue = sum(row["items"] * row["unit_price_cents"] for row in expected_orders)
assert int(revenue.revenue_cents.sum()) == expected_revenue
display(revenue.assign(revenue=revenue.revenue_cents / 100).drop(columns="revenue_cents"))
print("Recovered:", len(actual_orders), "unique orders with all original fields intact.")

delivery_02.json: 8 incoming lines


Query identity restored from checkpoint: True


,department,orders,units,revenue
0,Books,6,12,1500.0
1,Electronics,6,12,4668.0
2,Stationery,6,12,540.0


Recovered: 18 unique orders with all original fields intact.


In [8]:
before_repeat = read_table("SELECT * FROM orders ORDER BY order_id")
run_available()
after_repeat = read_table("SELECT * FROM orders ORDER BY order_id")
pd.testing.assert_frame_equal(before_repeat, after_repeat)

audit = pd.DataFrame([
    ("Expected unique orders", len(expected_orders)),
    ("Recovered unique orders", len(after_repeat)),
    ("Duplicate IDs in ledger", int(after_repeat.order_id.duplicated().sum())),
    ("Revenue in cents", int(revenue.revenue_cents.sum())),
    ("Batch IDs committed", len(read_table("SELECT batch_id FROM processed_batches")))
], columns=["check", "value"])
display(audit)
print("Restarting with no new input left the ledger unchanged.")
actual_orders.to_csv(root / "recovered_orders.csv", index=False)
revenue.to_csv(root / "department_totals.csv", index=False)
spark.stop()

,check,value
0,Expected unique orders,18
1,Recovered unique orders,18
2,Duplicate IDs in ledger,0
3,Revenue in cents,670800
4,Batch IDs committed,5


Restarting with no new input left the ledger unchanged.


## Result and limits

The assertions verify all 18 order records and their revenue after a failure and restart. Replay does not duplicate the external writes because the sink records batch IDs in the same transaction as the orders. Spark's checkpoint retains progress and de-duplication state; the original input files remain available for replay.

The demonstration assumes a single writer, unchanged event IDs and an unchanged query definition. The database and its matching checkpoint must be preserved together. Deleting the checkpoint while reusing the batch log would invalidate the batch-ID assumptions. The example's guarantees depend on the transaction and replay logic; `foreachBatch` alone does not guarantee exactly-once external writes.

Reference: [Apache Spark Structured Streaming guide](https://spark.apache.org/docs/4.0.1/streaming/apis-on-dataframes-and-datasets.html).